In [ ]:
import pandas as pd
import pandas.api.types
import numpy as np
import math

class ParticipantVisibleError(Exception):
    # If you want an error message to be shown to participants, you must raise the error as a ParticipantVisibleError.
    # All other errors will only be shown to the competition host.
    pass

def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    '''
    Compute the average IOU between the solution and submission labels.

    Each label is expected to be a flat list of 25 numbers representing 5 polygons
    (each polygon in the format [class, cx, cy, w, h]). Padded polygons should have
    a class of -1. The IOU is computed by first converting the polygons into a binary mask.

    Parameters:
        solution (pd.DataFrame): DataFrame with solution labels.
        submission (pd.DataFrame): DataFrame with submission labels.
        row_id_column_name (str): Name of the column used for row IDs.

    Returns:
        float: The average IOU score.
    '''
    # Remove the row ID columns.
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    # Check that every submission column is numeric.
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')

    # Nested helper functions.
    def create_polygon_mask(polygon, image_size=(64, 64)):
        """
        Create a binary mask for a single polygon.

        The polygon is specified as [class, cx, cy, w, h] with normalized
        coordinates (cx, cy, w, h) in [0,1].

        - For rectangles (class == 1), the mask is a filled rectangle.
        - For circles (class == 0), the mask is a filled circle (w or h is the diameter).

        Parameters:
            polygon (list or numpy array): [class, cx, cy, w, h]
            image_size (tuple): (height, width) of the output mask.

        Returns:
            mask (numpy array): A binary mask of shape (height, width).
        """
        height, width = image_size
        cls = polygon[0]
        cx, cy, w, h = polygon[1:5]

        # Create an empty mask.
        mask = np.zeros((height, width), dtype=np.float32)

        if cls == 1:  # Rectangle
            # Convert normalized coordinates to pixel coordinates.
            cx_pix = cx * width
            cy_pix = cy * height
            half_w = (w * width) / 2.0
            half_h = (h * height) / 2.0

            x_min = int(max(cx_pix - half_w, 0))
            x_max = int(min(cx_pix + half_w, width))
            y_min = int(max(cy_pix - half_h, 0))
            y_max = int(min(cy_pix + half_h, height))

            mask[y_min:y_max, x_min:x_max] = 1.0

        elif cls == 0:  # Circle
            # For circles, w (or h) is the diameter.
            cx_pix = cx * width
            cy_pix = cy * height
            radius = (w * width) / 2.0  # using width as diameter

            # Create a grid of (x,y) coordinates.
            ys = np.arange(height).reshape(height, 1)
            xs = np.arange(width).reshape(1, width)

            # Compute the Euclidean distance from each pixel to the center.
            dist = np.sqrt((xs - cx_pix) ** 2 + (ys - cy_pix) ** 2)
            mask[dist <= radius] = 1.0

        else:
            raise ValueError("Invalid polygon class. Expected 0 for circle or 1 for rectangle.")

        return mask

    def create_mask_from_label(label, image_size=(64, 64)):
        """
        Create a single binary mask from a label that contains 5 polygons.

        The label is expected to be a list or numpy array with 25 numbers
        ([poly1, poly2, poly3, poly4, poly5]) where each polygon is defined as
        [class, cx, cy, w, h]. Polygons with class == -1 are ignored.

        The final mask is the union (logical OR) of the masks of the valid polygons.

        Parameters:
            label (list or numpy array): 1D array of length 25.
            image_size (tuple): (height, width) for the mask.

        Returns:
            combined_mask (numpy array): Binary mask of shape (height, width).
        """
        # Convert label to a NumPy array (if not already) and reshape to (num_polygons, 5)
        label_arr = np.array(label) if not isinstance(label, np.ndarray) else label
        polygons = label_arr.reshape(-1, 5)

        height, width = image_size
        combined_mask = np.zeros((height, width), dtype=np.float32)

        for poly in polygons:
            if poly[0] == -1:
                continue  # Skip padded polygons.
            poly_mask = create_polygon_mask(poly, image_size)
            # Combine masks via logical OR (by summing and clipping to 1).
            combined_mask = np.clip(combined_mask + poly_mask, 0, 1)

        return combined_mask

    def compute_iou(mask1, mask2, eps=1e-6):
        """
        Compute the Intersection over Union (IOU) between two binary masks.

        Parameters:
            mask1, mask2 (numpy arrays): Binary masks of the same shape.

        Returns:
            iou (float): The IOU value.
        """
        intersection = np.sum((mask1 > 0) & (mask2 > 0))
        union = np.sum((mask1 > 0) | (mask2 > 0))
        iou = intersection / (union + eps)
        return iou

    def compute_iou_from_labels(pred_label, target_label, image_size=(64, 64)):
        """
        Compute the IOU between two labels, where each label contains 5 polygons.

        Each label is a 1D list or numpy array of length 25 (5 polygons * 5 numbers each).
        The function converts each label into a single combined mask and then computes the IOU.

        Parameters:
            pred_label, target_label (list or numpy array): Each in shape (25,)
            image_size (tuple): (height, width) of the mask.

        Returns:
            iou (float): The IOU between the combined masks.
        """
        mask_pred = create_mask_from_label(pred_label, image_size)
        mask_target = create_mask_from_label(target_label, image_size)
        return compute_iou(mask_pred, mask_target)

    def average_iou(pred_labels, target_labels, image_size=(64, 64)):
        """
        Compute the average IOU over a batch of labels.

        Each element in pred_labels and target_labels is expected to be a 1D list or numpy array
        of length 25 (i.e. 5 polygons per label). Labels that have all polygons padded
        (class == -1) are ignored.

        Parameters:
            pred_labels (numpy array): Array of shape (N, 25) for N labels.
            target_labels (numpy array): Array of shape (N, 25) for N labels.
            image_size (tuple): (height, width) for the masks.

        Returns:
            avg_iou (float): The average IOU over valid labels.
        """
        total_iou = 0.0
        valid_count = 0

        for i in range(pred_labels.shape[0]):
            label_target = target_labels[i]
            polygons = label_target.reshape(-1, 5)
            if np.all(polygons[:, 0] == -1):
                continue  # Skip labels with all padded polygons.

            pred_label = pred_labels[i]
            iou = compute_iou_from_labels(pred_label, label_target, image_size)
            total_iou += iou
            valid_count += 1

        if valid_count == 0:
            return 0.0
        return total_iou / valid_count

    # Convert the solution and submission DataFrames to NumPy arrays.
    sol_arr = solution.to_numpy()
    sub_arr = submission.to_numpy()

    return average_iou(sol_arr, sub_arr, image_size=(64, 64))
